# 09 — Advanced Candidate Generation

## 1. Objective

The current recommendation system only considers products that a customer has purchased before.

This works well for reorder prediction, but it cannot recommend new products that the customer may also like.

The objective of this notebook is to expand the candidate set by adding new-to-customer products using signals such as:

- Customer aisle preferences
- Customer department preferences
- Product popularity
- Co-purchase relationships

The expanded candidate set will later allow the recommendation system to move from pure reorder prediction toward broader next-basket prediction.

## 2. Load Source Data

Advanced candidate generation must use only information available before the customer's target order.

We load:

- Customer orders
- Products contained in each order
- Product metadata, including aisle and department

These tables will be used to analyze historical customer preferences and discover additional candidate products.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load cleaned source tables
orders_df = spark.table("workspace.cleaned_data.orders")

order_products_df = spark.table(
    "workspace.cleaned_data.order_products"
)

products_df = (
    spark.table("workspace.cleaned_data.products")
    .select(
        "product_id",
        "product_name",
        "aisle_id",
        "department_id"
    )
)

print("Source tables loaded successfully.")

Source tables loaded successfully.


## 3. Define Historical and Target Orders

For each customer:

- `prior` orders are used as historical information.
- the `train` order is the target basket we want to predict.

Only historical orders will be used to generate new product candidates.

In [0]:
# Target order for each customer
target_orders_df = (
    orders_df
    .filter(F.col("eval_set") == "train")
    .select(
        F.col("order_id").alias("target_order_id"),
        "user_id",
        F.col("order_number").alias("target_order_number")
    )
)

# Historical orders only
historical_orders_df = (
    orders_df
    .filter(F.col("eval_set") == "prior")
    .join(
        target_orders_df.select("user_id"),
        on="user_id",
        how="inner"
    )
    .select(
        "order_id",
        "user_id",
        "order_number"
    )
)

print("Target orders:", target_orders_df.count())
print("Historical orders:", historical_orders_df.count())

Target orders: 131209
Historical orders: 2047377


## 4. Build Historical Purchase Transactions

We combine historical orders with the products purchased in those orders.

We also attach each product's aisle and department so that we can later identify category preferences and discover new candidate products.

In [0]:
historical_transactions_df = (
    historical_orders_df
    .join(
        order_products_df.select(
            "order_id",
            "product_id"
        ),
        on="order_id",
        how="inner"
    )
    .join(
        products_df,
        on="product_id",
        how="left"
    )
    .select(
        "user_id",
        "order_id",
        "order_number",
        "product_id",
        "product_name",
        "aisle_id",
        "department_id"
    )
)

print(
    "Historical transactions:",
    historical_transactions_df.count()
)

Historical transactions: 20641991


## 5. Identify Previously Purchased Products

We create one unique customer-product pair for every product the customer has purchased historically.

This table will be used to distinguish existing reorder candidates from genuinely new product recommendations.

In [0]:
historical_user_products_df = (
    historical_transactions_df
    .select(
        "user_id",
        "product_id"
    )
    .distinct()
)

print(
    "Unique historical customer-product pairs:",
    historical_user_products_df.count()
)

Unique historical customer-product pairs: 8474661


## 6. Identify Customer Aisle Preferences

We measure how strongly each customer prefers each aisle based on their historical purchases.

Customers' most frequently purchased aisles will later be used to generate relevant new-product candidates.

In [0]:
# Count historical purchases by customer and aisle
customer_aisle_preferences_df = (
    historical_transactions_df
    .groupBy(
        "user_id",
        "aisle_id"
    )
    .agg(
        F.count("*").alias("customer_aisle_purchases")
    )
)

# Rank aisles for each customer
aisle_rank_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.desc("customer_aisle_purchases"),
        F.asc("aisle_id")
    )
)

customer_aisle_preferences_df = (
    customer_aisle_preferences_df
    .withColumn(
        "aisle_rank",
        F.row_number().over(aisle_rank_window)
    )
)

display(
    customer_aisle_preferences_df
    .filter(F.col("aisle_rank") <= 5)
    .orderBy(
        "user_id",
        "aisle_rank"
    )
    .limit(25)
)

user_id,aisle_id,customer_aisle_purchases,aisle_rank
1,77,13,1
1,23,12,2
1,117,9,3
1,21,8,4
1,24,5,5
2,120,42,1
2,24,33,2
2,38,12,3
2,107,12,4
2,78,11,5


## 7. Identify Popular Products Within Each Aisle

We identify the most popular products inside each aisle using historical purchases.

Product popularity is primarily measured by the number of unique customers who purchased the product.

These popular products will later be matched with each customer's favorite aisles to create new-product candidates.

In [0]:
# Calculate historical product popularity within each aisle
aisle_product_popularity_df = (
    historical_transactions_df
    .groupBy(
        "aisle_id",
        "product_id",
        "product_name"
    )
    .agg(
        F.countDistinct("user_id").alias("unique_customers"),
        F.count("*").alias("total_purchases")
    )
)

# Rank products within each aisle
product_rank_window = (
    Window
    .partitionBy("aisle_id")
    .orderBy(
        F.desc("unique_customers"),
        F.desc("total_purchases"),
        F.asc("product_id")
    )
)

aisle_product_popularity_df = (
    aisle_product_popularity_df
    .withColumn(
        "product_rank_in_aisle",
        F.row_number().over(product_rank_window)
    )
)

# Display the top 10 products from a few aisles
display(
    aisle_product_popularity_df
    .filter(F.col("product_rank_in_aisle") <= 10)
    .orderBy(
        "aisle_id",
        "product_rank_in_aisle"
    )
    .limit(30)
)

aisle_id,product_id,product_name,unique_customers,total_purchases,product_rank_in_aisle
1,22281,Chicken Noodle Soup,1464,2760,1
1,25199,Classic Chicken Salad,1048,3183,2
1,26047,Tuna Salad,904,3818,3
1,23719,Chicken Tortilla Soup,903,1730,4
1,21560,Cut Hearts Of Palm,889,2401,5
1,8382,Organic Tomato Bisque,469,801,6
1,29180,Tuscan Kale & Quinoa Salad,457,1535,7
1,43221,Egg Salad,454,1362,8
1,25965,Santa Fe Fiesta Salad,411,1430,9
1,26714,Chicken Salad,404,1260,10


## 8. Generate New-Product Candidates from Favorite Aisles

We combine each customer's favorite aisles with popular products from those aisles.

Products that the customer has already purchased are removed.

The remaining products represent new-to-customer candidates that may be relevant based on the customer's historical aisle preferences.

In [0]:
# Keep each customer's top 5 favorite aisles
top_customer_aisles_df = (
    customer_aisle_preferences_df
    .filter(F.col("aisle_rank") <= 5)
    .select(
        "user_id",
        "aisle_id",
        "aisle_rank"
    )
)

# Keep the top 10 popular products in each aisle
top_aisle_products_df = (
    aisle_product_popularity_df
    .filter(F.col("product_rank_in_aisle") <= 10)
    .select(
        "aisle_id",
        "product_id",
        "product_name",
        "product_rank_in_aisle"
    )
)

# Match customers with popular products from their favorite aisles
aisle_candidates_df = (
    top_customer_aisles_df
    .join(
        top_aisle_products_df,
        on="aisle_id",
        how="inner"
    )
)

# Remove products the customer already purchased
new_aisle_candidates_df = (
    aisle_candidates_df
    .join(
        historical_user_products_df,
        on=["user_id", "product_id"],
        how="left_anti"
    )
)

display(
    new_aisle_candidates_df
    .select(
        "user_id",
        "product_id",
        "product_name",
        "aisle_id",
        "aisle_rank",
        "product_rank_in_aisle"
    )
    .orderBy(
        "user_id",
        "aisle_rank",
        "product_rank_in_aisle"
    )
    .limit(30)
)

user_id,product_id,product_name,aisle_id,aisle_rank,product_rank_in_aisle
1,12916,Ginger Ale,77,1,2
1,10957,Fridge Pack Cola,77,1,3
1,47141,Cola,77,1,4
1,16696,Coke Classic,77,1,5
1,40910,Root Beer,77,1,6
1,4138,Arancita Rossa,77,1,7
1,24759,Club Soda,77,1,8
1,44375,Canned Aranciata Orange,77,1,9
1,16290,Sparkling Clementine Juice,77,1,10
1,3599,Baked Aged White Cheddar Rice and Corn Puffs,23,2,1


### 8.1 — Measure Aisle-Based Candidate Generation

We measure how many new customer-product candidates were generated from favorite aisles and how many customers received at least one new candidate.

This helps us control the size of the expanded candidate set before adding additional generation strategies.

In [0]:
aisle_candidate_summary_df = (
    new_aisle_candidates_df
    .agg(
        F.count("*").alias("new_candidate_pairs"),
        F.countDistinct("user_id").alias("customers_with_candidates"),
        F.countDistinct("product_id").alias("unique_candidate_products")
    )
)

display(aisle_candidate_summary_df)

new_candidate_pairs,customers_with_candidates,unique_candidate_products
5526136,131209,1340


### 8.2 — Measure New-Product Coverage

We evaluate whether the aisle-based candidates recover products that customers actually purchased for the first time in their target basket.

This measures whether advanced candidate generation improves our ability to predict new-to-customer products.

In [0]:
# Products actually purchased in each target order
target_products_df = (
    target_orders_df
    .select(
        "user_id",
        "target_order_id"
    )
    .join(
        order_products_df.select(
            F.col("order_id").alias("target_order_id"),
            "product_id"
        ),
        on="target_order_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
)

# Keep only products that were NEW to the customer
target_new_products_df = (
    target_products_df
    .join(
        historical_user_products_df,
        on=["user_id", "product_id"],
        how="left_anti"
    )
)

# Check which new target products were recovered
recovered_new_products_df = (
    target_new_products_df
    .join(
        new_aisle_candidates_df.select(
            "user_id",
            "product_id"
        ).distinct(),
        on=["user_id", "product_id"],
        how="inner"
    )
)

# Calculate coverage
target_new_count = target_new_products_df.count()
recovered_new_count = recovered_new_products_df.count()

new_product_coverage = (
    recovered_new_count / target_new_count * 100
)

print("Actual new products in target baskets:", target_new_count)
print("Recovered by aisle candidates:", recovered_new_count)
print(
    "New-product coverage:",
    round(new_product_coverage, 2),
    "%"
)

Actual new products in target baskets: 555793
Recovered by aisle candidates: 36728
New-product coverage: 6.61 %


### 8.3 — Measure Overall Candidate Coverage Improvement

The original candidate-generation strategy included only products previously purchased by each customer.

We now combine:

- Historical reorder candidates
- New aisle-based candidates

We compare the expanded candidate set with the original baseline to measure how much additional target-basket coverage was gained.

In [0]:
# Baseline candidate coverage
baseline_candidates_df = (
    historical_user_products_df
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id"
        ),
        on="user_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
)

# Add target order ID to new aisle candidates
new_aisle_candidates_with_target_df = (
    new_aisle_candidates_df
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id"
        ),
        on="user_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
)

# Combine reorder + new-product candidates
expanded_candidates_df = (
    baseline_candidates_df
    .unionByName(new_aisle_candidates_with_target_df)
    .distinct()
)

# Products in actual target baskets
total_target_products = target_products_df.count()

# Baseline products recovered
baseline_recovered = (
    target_products_df
    .join(
        baseline_candidates_df,
        on=["user_id", "target_order_id", "product_id"],
        how="inner"
    )
    .count()
)

# Expanded products recovered
expanded_recovered = (
    target_products_df
    .join(
        expanded_candidates_df,
        on=["user_id", "target_order_id", "product_id"],
        how="inner"
    )
    .count()
)

baseline_coverage = baseline_recovered / total_target_products * 100
expanded_coverage = expanded_recovered / total_target_products * 100

print("Baseline coverage:", round(baseline_coverage, 2), "%")
print("Expanded coverage:", round(expanded_coverage, 2), "%")
print(
    "Coverage improvement:",
    round(expanded_coverage - baseline_coverage, 2),
    "percentage points"
)

Baseline coverage: 59.86 %
Expanded coverage: 62.51 %
Coverage improvement: 2.65 percentage points


## 9. Co-Purchase Candidate Generation

Favorite-aisle candidates use category preferences, but they do not capture relationships between specific products.

We now analyze products that historically appear together in the same basket.

The idea is:

**If a customer previously purchased Product A, products that are frequently purchased together with Product A may also be relevant candidates.**

We begin by creating unique order-product pairs from historical transactions.

In [0]:
historical_order_products_df = (
    historical_transactions_df
    .select(
        "order_id",
        "user_id",
        "product_id"
    )
    .distinct()
)

print(
    "Historical order-product pairs:",
    historical_order_products_df.count()
)

Historical order-product pairs: 20641991


### 9.1 — Inspect Historical Basket Sizes

Co-purchase analysis compares products that appear together in the same order.

Because very large baskets can generate a very large number of product pairs, we first inspect historical basket sizes before calculating co-purchase relationships.

In [0]:
# Calculate the number of products in each historical order
historical_basket_sizes_df = (
    historical_order_products_df
    .groupBy("order_id")
    .agg(
        F.count("product_id").alias("basket_size")
    )
)

# Summarize basket sizes
basket_size_summary_df = (
    historical_basket_sizes_df
    .agg(
        F.count("*").alias("historical_orders"),
        F.round(F.avg("basket_size"), 2).alias("avg_basket_size"),
        F.max("basket_size").alias("max_basket_size"),
        F.expr(
            "percentile_approx(basket_size, array(0.5, 0.9, 0.95, 0.99))"
        ).alias("basket_size_percentiles"),
        F.sum(
            F.when(F.col("basket_size") > 50, 1).otherwise(0)
        ).alias("orders_above_50_products")
    )
)

display(basket_size_summary_df)

historical_orders,avg_basket_size,max_basket_size,basket_size_percentiles,orders_above_50_products
2047377,10.08,145,"List(8, 20, 25, 35)",1953


### 9.2 — Limit Very Large Baskets

Co-purchase analysis creates combinations between products appearing in the same basket.

Very large baskets can generate an unusually large number of product pairs and significantly increase computation.

Since more than 99% of historical orders contain 50 products or fewer, we exclude baskets above 50 products from the co-purchase calculation.

In [0]:
# Keep orders with 50 products or fewer
eligible_copurchase_orders_df = (
    historical_basket_sizes_df
    .filter(F.col("basket_size") <= 50)
    .select("order_id")
)

# Keep products belonging to those orders
copurchase_order_products_df = (
    historical_order_products_df
    .join(
        eligible_copurchase_orders_df,
        on="order_id",
        how="inner"
    )
)

print(
    "Orders used for co-purchase analysis:",
    eligible_copurchase_orders_df.count()
)

print(
    "Order-product pairs used:",
    copurchase_order_products_df.count()
)

Orders used for co-purchase analysis: 2045424
Order-product pairs used: 20527503


### 9.3 — Build Historical Co-Purchase Pairs

We identify pairs of products that appeared together in the same historical basket.

For each product pair, we count how many different baskets contained both products.

Pairs that frequently occur together can later be used to suggest new products related to items a customer has previously purchased.

In [0]:
# Create two copies of the historical order-product data
products_a_df = (
    copurchase_order_products_df
    .select(
        "order_id",
        F.col("product_id").alias("product_a")
    )
)

products_b_df = (
    copurchase_order_products_df
    .select(
        "order_id",
        F.col("product_id").alias("product_b")
    )
)

# Create unique product pairs within the same basket
pair_occurrences_df = (
    products_a_df
    .join(
        products_b_df,
        on="order_id",
        how="inner"
    )
    .filter(
        F.col("product_a") < F.col("product_b")
    )
    .select(
        "product_a",
        "product_b"
    )
)

# Count how many baskets contain each product pair
copurchase_pairs_df = (
    pair_occurrences_df
    .groupBy(
        "product_a",
        "product_b"
    )
    .agg(
        F.count("*").alias("copurchase_orders")
    )
)

# Show the most frequent co-purchase relationships
display(
    copurchase_pairs_df
    .orderBy(F.desc("copurchase_orders"))
    .limit(20)
)

product_a,product_b,copurchase_orders
13176,47209,39536
13176,21137,38611
21137,24852,35764
24852,47766,34090
21903,24852,32237
13176,21903,31942
16797,24852,26078
21137,47209,25876
13176,27966,25492
24852,47626,25453


### 9.4 — Calculate Product Basket Frequency

Raw co-purchase counts can be influenced by overall product popularity.

We therefore calculate how many historical baskets contain each product.

These counts will later be used to calculate stronger product-association measures such as confidence and lift.

In [0]:
# Count how many eligible historical baskets contain each product
product_basket_frequency_df = (
    copurchase_order_products_df
    .groupBy("product_id")
    .agg(
        F.countDistinct("order_id").alias("product_basket_count")
    )
)

display(
    product_basket_frequency_df
    .orderBy(F.desc("product_basket_count"))
    .limit(20)
)

product_id,product_basket_count
24852,300181
13176,239144
21137,168016
21903,153652
47209,135922
47766,113024
47626,96875
16797,90228
26209,89004
27845,87741


### 9.5 — Measure Co-Purchase Relationship Strength

Raw co-purchase frequency does not necessarily mean that two products have a strong relationship because very popular products naturally appear in many baskets.

We therefore calculate:

- **Confidence** — among baskets containing Product A, how often does Product B also appear?
- **Lift** — how much more frequently A and B appear together than we would expect based on their individual popularity.

A lift greater than `1` indicates a positive association between the two products.

In [0]:
# Total number of historical baskets used
total_copurchase_orders = eligible_copurchase_orders_df.count()

# Create directional relationships:
# A -> B
rules_ab_df = (
    copurchase_pairs_df
    .select(
        F.col("product_a").alias("source_product_id"),
        F.col("product_b").alias("related_product_id"),
        "copurchase_orders"
    )
)

# B -> A
rules_ba_df = (
    copurchase_pairs_df
    .select(
        F.col("product_b").alias("source_product_id"),
        F.col("product_a").alias("related_product_id"),
        "copurchase_orders"
    )
)

# Combine both directions
directional_pairs_df = (
    rules_ab_df
    .unionByName(rules_ba_df)
)

# Add basket frequencies for source and related products
copurchase_strength_df = (
    directional_pairs_df

    .join(
        product_basket_frequency_df
        .withColumnRenamed("product_id", "source_product_id")
        .withColumnRenamed(
            "product_basket_count",
            "source_basket_count"
        ),
        on="source_product_id",
        how="inner"
    )

    .join(
        product_basket_frequency_df
        .withColumnRenamed("product_id", "related_product_id")
        .withColumnRenamed(
            "product_basket_count",
            "related_basket_count"
        ),
        on="related_product_id",
        how="inner"
    )

    .withColumn(
        "confidence",
        F.col("copurchase_orders") /
        F.col("source_basket_count")
    )

    .withColumn(
        "lift",
        (
            F.col("copurchase_orders") * F.lit(total_copurchase_orders)
        ) / (
            F.col("source_basket_count") *
            F.col("related_basket_count")
        )
    )
)

display(
    copurchase_strength_df
    .filter(F.col("copurchase_orders") >= 100)
    .orderBy(F.desc("lift"))
    .select(
        "source_product_id",
        "related_product_id",
        "copurchase_orders",
        F.round("confidence", 4).alias("confidence"),
        F.round("lift", 2).alias("lift")
    )
    .limit(20)
)

source_product_id,related_product_id,copurchase_orders,confidence,lift
9055,17573,100,0.3953,5575.64
17573,9055,100,0.6897,5575.64
3858,15692,158,0.4788,4971.19
15692,3858,158,0.802,4971.19
20153,46949,113,0.4768,4851.96
46949,20153,113,0.5622,4851.96
45562,25876,114,0.6,4631.15
25876,45562,114,0.4302,4631.15
40292,34456,100,0.5128,4620.86
34456,40292,100,0.4405,4620.86


### 9.6 — Filter Reliable Co-Purchase Relationships

Very rare products can produce extremely high lift values even when the relationship is based on relatively few baskets.

To create more reliable recommendation rules, we keep relationships with sufficient historical evidence.

We require:

- At least 200 baskets containing both products
- At least 1,000 historical baskets containing each individual product
- Confidence of at least 2%
- Lift greater than 1

These thresholds reduce noisy associations while preserving meaningful co-purchase relationships.

In [0]:
reliable_copurchase_df = (
    copurchase_strength_df
    .filter(
        (F.col("copurchase_orders") >= 200) &
        (F.col("source_basket_count") >= 1000) &
        (F.col("related_basket_count") >= 1000) &
        (F.col("confidence") >= 0.02) &
        (F.col("lift") > 1)
    )
)

display(
    reliable_copurchase_df
    .orderBy(
        F.desc("lift"),
        F.desc("confidence"),
        F.desc("copurchase_orders")
    )
    .select(
        "source_product_id",
        "related_product_id",
        "copurchase_orders",
        "source_basket_count",
        "related_basket_count",
        F.round("confidence", 4).alias("confidence"),
        F.round("lift", 2).alias("lift")
    )
    .limit(20)
)

source_product_id,related_product_id,copurchase_orders,source_basket_count,related_basket_count,confidence,lift
35050,1577,545,1185,1192,0.4599,789.2
1577,35050,545,1192,1185,0.4572,789.2
21527,35050,451,1026,1185,0.4396,758.74
35050,21527,451,1185,1026,0.3806,758.74
21527,1577,445,1026,1192,0.4337,744.25
1577,21527,445,1192,1026,0.3733,744.25
44786,13269,979,1495,1878,0.6548,713.23
13269,44786,979,1878,1495,0.5213,713.23
10036,12020,664,1313,1527,0.5057,677.4
12020,10036,664,1527,1313,0.4348,677.4


### 9.7 — Rank the Strongest Co-Purchase Relationships

A product can have many historical co-purchase relationships.

To keep candidate generation focused and computationally manageable, we rank related products for each source product.

Relationships are prioritized using:

1. Confidence
2. Lift
3. Number of co-purchase baskets

We retain the Top 5 related products for each source product.

In [0]:
# Rank related products for every source product
copurchase_rank_window = (
    Window
    .partitionBy("source_product_id")
    .orderBy(
        F.desc("confidence"),
        F.desc("lift"),
        F.desc("copurchase_orders"),
        F.asc("related_product_id")
    )
)

top_copurchase_rules_df = (
    reliable_copurchase_df
    .withColumn(
        "related_product_rank",
        F.row_number().over(copurchase_rank_window)
    )
    .filter(
        F.col("related_product_rank") <= 5
    )
)

# Add product names for easier interpretation
source_names_df = (
    products_df
    .select(
        F.col("product_id").alias("source_product_id"),
        F.col("product_name").alias("source_product_name")
    )
)

related_names_df = (
    products_df
    .select(
        F.col("product_id").alias("related_product_id"),
        F.col("product_name").alias("related_product_name")
    )
)

top_copurchase_rules_named_df = (
    top_copurchase_rules_df
    .join(
        source_names_df,
        on="source_product_id",
        how="left"
    )
    .join(
        related_names_df,
        on="related_product_id",
        how="left"
    )
)

display(
    top_copurchase_rules_named_df
    .select(
        "source_product_id",
        "source_product_name",
        "related_product_id",
        "related_product_name",
        "copurchase_orders",
        F.round("confidence", 4).alias("confidence"),
        F.round("lift", 2).alias("lift"),
        "related_product_rank"
    )
    .orderBy(
        "source_product_id",
        "related_product_rank"
    )
    .limit(30)
)

source_product_id,source_product_name,related_product_id,related_product_name,copurchase_orders,confidence,lift,related_product_rank
10,Sparkling Orange Juice & Prickly Pear Beverage,4138,Arancita Rossa,381,0.2336,107.32,1
10,Sparkling Orange Juice & Prickly Pear Beverage,24852,Banana,290,0.1778,1.21,2
10,Sparkling Orange Juice & Prickly Pear Beverage,44375,Canned Aranciata Orange,221,0.1355,72.14,3
10,Sparkling Orange Juice & Prickly Pear Beverage,33198,Sparkling Natural Mineral Water,200,0.1226,9.05,4
25,Salted Caramel Lean Protein & Fiber Bar,24852,Banana,340,0.2686,1.83,1
25,Salted Caramel Lean Protein & Fiber Bar,13176,Bag of Organic Bananas,212,0.1675,1.43,2
34,Peanut Butter Cereal,24852,Banana,1147,0.2742,1.87,1
34,Peanut Butter Cereal,13176,Bag of Organic Bananas,606,0.1449,1.24,2
34,Peanut Butter Cereal,21137,Organic Strawberries,534,0.1277,1.55,3
34,Peanut Butter Cereal,21903,Organic Baby Spinach,435,0.104,1.38,4


### 9.8 — Generate Personalized Co-Purchase Candidates

We apply the strongest co-purchase relationships to each customer's historical purchases.

If a product previously purchased by the customer is strongly associated with another product, that related product becomes a potential candidate.

Products already purchased by the customer are removed so that the resulting candidates are genuinely new-to-customer products.

When several historical products suggest the same candidate, we retain the strongest relationship and count how many source products supported the recommendation.

In [0]:
# Match each customer's historical products with co-purchase rules
customer_copurchase_matches_df = (
    historical_user_products_df
    .withColumnRenamed("product_id", "source_product_id")
    .join(
        top_copurchase_rules_df,
        on="source_product_id",
        how="inner"
    )
)

# Aggregate when several purchased products suggest the same new product
copurchase_candidates_df = (
    customer_copurchase_matches_df
    .groupBy(
        "user_id",
        F.col("related_product_id").alias("product_id")
    )
    .agg(
        F.countDistinct("source_product_id")
        .alias("supporting_source_products"),

        F.max("confidence")
        .alias("max_confidence"),

        F.max("lift")
        .alias("max_lift"),

        F.max("copurchase_orders")
        .alias("max_copurchase_orders")
    )
)

# Remove products the customer already purchased
new_copurchase_candidates_df = (
    copurchase_candidates_df
    .join(
        historical_user_products_df,
        on=["user_id", "product_id"],
        how="left_anti"
    )
    .join(
        products_df.select(
            "product_id",
            "product_name"
        ),
        on="product_id",
        how="left"
    )
)

display(
    new_copurchase_candidates_df
    .select(
        "user_id",
        "product_id",
        "product_name",
        "supporting_source_products",
        F.round("max_confidence", 4).alias("max_confidence"),
        F.round("max_lift", 2).alias("max_lift"),
        "max_copurchase_orders"
    )
    .orderBy(
        "user_id",
        F.desc("supporting_source_products"),
        F.desc("max_confidence")
    )
    .limit(30)
)

user_id,product_id,product_name,supporting_source_products,max_confidence,max_lift,max_copurchase_orders
1,6184,Clementines,9,0.2185,22.77,2168
1,24852,Banana,6,0.3891,2.65,8720
1,37710,Trail Mix,6,0.185,51.24,1486
1,21137,Organic Strawberries,6,0.174,2.12,38611
1,47209,Organic Hass Avocado,5,0.1653,2.49,39536
1,21903,Organic Baby Spinach,4,0.1883,2.51,31942
1,16797,Strawberries,2,0.1663,3.77,2708
1,12341,Hass Avocados,2,0.1391,8.91,602
1,43352,Raspberries,2,0.1246,7.02,724
1,47766,Organic Avocado,2,0.1241,2.25,2780


### 9.9 — Measure Co-Purchase Candidate Generation

We summarize the personalized co-purchase candidate set.

We measure:

- The total number of new customer-product candidates
- The number of customers receiving at least one candidate
- The number of unique products represented

This helps us understand the size and reach of the co-purchase strategy before evaluating its target-basket coverage.

In [0]:
copurchase_candidate_summary_df = (
    new_copurchase_candidates_df
    .agg(
        F.count("*").alias("new_candidate_pairs"),
        F.countDistinct("user_id").alias("customers_with_candidates"),
        F.countDistinct("product_id").alias("unique_candidate_products")
    )
)

display(copurchase_candidate_summary_df)

new_candidate_pairs,customers_with_candidates,unique_candidate_products
2507652,130943,605


### 9.10 — Measure Co-Purchase New-Product Coverage

We evaluate whether the personalized co-purchase candidates recover products that customers actually purchased for the first time in their target basket.

This allows us to compare the co-purchase strategy with the earlier favorite-aisle strategy.

In [0]:
# Check which actual new target products are recovered
# by the co-purchase candidate strategy
recovered_copurchase_new_products_df = (
    target_new_products_df
    .join(
        new_copurchase_candidates_df
        .select(
            "user_id",
            "product_id"
        )
        .distinct(),
        on=["user_id", "product_id"],
        how="inner"
    )
)

# Count recovered products
recovered_copurchase_new_count = (
    recovered_copurchase_new_products_df.count()
)

# Calculate coverage
copurchase_new_product_coverage = (
    recovered_copurchase_new_count
    / target_new_count
    * 100
)

print(
    "Actual new products in target baskets:",
    target_new_count
)

print(
    "Recovered by co-purchase candidates:",
    recovered_copurchase_new_count
)

print(
    "Co-purchase new-product coverage:",
    round(copurchase_new_product_coverage, 2),
    "%"
)

Actual new products in target baskets: 555793
Recovered by co-purchase candidates: 33190
Co-purchase new-product coverage: 5.97 %


### 9.11 — Combine Advanced Candidate Strategies

The aisle-based and co-purchase strategies may recover different new products.

We therefore combine both candidate sets and remove duplicate customer-product pairs.

We then measure the new-product coverage of the combined strategy.

In [0]:
# Aisle-based new candidates
aisle_new_pairs_df = (
    new_aisle_candidates_df
    .select(
        "user_id",
        "product_id"
    )
    .distinct()
)

# Co-purchase new candidates
copurchase_new_pairs_df = (
    new_copurchase_candidates_df
    .select(
        "user_id",
        "product_id"
    )
    .distinct()
)

# Combine both strategies
combined_new_candidates_df = (
    aisle_new_pairs_df
    .unionByName(copurchase_new_pairs_df)
    .distinct()
)

# Find actual new target products recovered
recovered_combined_new_products_df = (
    target_new_products_df
    .join(
        combined_new_candidates_df,
        on=["user_id", "product_id"],
        how="inner"
    )
)

recovered_combined_count = (
    recovered_combined_new_products_df.count()
)

combined_new_product_coverage = (
    recovered_combined_count
    / target_new_count
    * 100
)

print(
    "Actual new products in target baskets:",
    target_new_count
)

print(
    "Recovered by combined strategies:",
    recovered_combined_count
)

print(
    "Combined new-product coverage:",
    round(combined_new_product_coverage, 2),
    "%"
)

Actual new products in target baskets: 555793
Recovered by combined strategies: 53086
Combined new-product coverage: 9.55 %


### 9.12 — Measure Final Overall Candidate Coverage

We combine the original reorder candidates with all advanced new-product candidates.

This represents the complete candidate-generation system developed in this notebook.

We compare its target-basket coverage with the original reorder-only baseline.

In [0]:
# Add target order IDs to the combined new-product candidates
combined_new_candidates_with_target_df = (
    combined_new_candidates_df
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id"
        ),
        on="user_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
)

# Combine baseline reorder candidates + advanced new-product candidates
final_candidate_set_df = (
    baseline_candidates_df
    .unionByName(combined_new_candidates_with_target_df)
    .distinct()
)

# Measure target-basket products recovered
final_recovered_count = (
    target_products_df
    .join(
        final_candidate_set_df,
        on=[
            "user_id",
            "target_order_id",
            "product_id"
        ],
        how="inner"
    )
    .count()
)

final_candidate_coverage = (
    final_recovered_count
    / total_target_products
    * 100
)

print("Baseline coverage:", round(baseline_coverage, 2), "%")
print("Final candidate coverage:", round(final_candidate_coverage, 2), "%")
print(
    "Total improvement:",
    round(final_candidate_coverage - baseline_coverage, 2),
    "percentage points"
)

Baseline coverage: 59.86 %
Final candidate coverage: 63.69 %
Total improvement: 3.83 percentage points


## 10. Evaluate Final Candidate Set Size

Increasing candidate coverage is useful only if the candidate set remains computationally manageable.

We therefore measure:

- Total customer-product candidates
- Number of customers covered
- Number of unique candidate products
- Average number of candidates per customer

This allows us to evaluate the trade-off between candidate-set size and target-basket coverage.

In [0]:
final_candidate_summary_df = (
    final_candidate_set_df
    .agg(
        F.count("*").alias("total_candidate_pairs"),
        F.countDistinct("user_id").alias("unique_customers"),
        F.countDistinct("product_id").alias("unique_products")
    )
)

final_candidate_summary_df = (
    final_candidate_summary_df
    .withColumn(
        "avg_candidates_per_customer",
        F.round(
            F.col("total_candidate_pairs")
            / F.col("unique_customers"),
            2
        )
    )
)

display(final_candidate_summary_df)

total_candidate_pairs,unique_customers,unique_products,avg_candidates_per_customer
15665871,131209,49468,119.4


### 10.1 — Rank Aisle-Based New Candidates

The aisle strategy can generate many new products for each customer.

We rank these candidates so that products from the customer's strongest aisle preferences and the most popular products within those aisles receive higher priority.

This ranking will later help us control the number of new candidates retained per customer.

In [0]:
aisle_candidate_rank_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.asc("aisle_rank"),
        F.asc("product_rank_in_aisle"),
        F.asc("product_id")
    )
)

ranked_aisle_candidates_df = (
    new_aisle_candidates_df
    .withColumn(
        "aisle_candidate_rank",
        F.row_number().over(aisle_candidate_rank_window)
    )
)

display(
    ranked_aisle_candidates_df
    .select(
        "user_id",
        "product_id",
        "product_name",
        "aisle_rank",
        "product_rank_in_aisle",
        "aisle_candidate_rank"
    )
    .orderBy(
        "user_id",
        "aisle_candidate_rank"
    )
    .limit(30)
)

user_id,product_id,product_name,aisle_rank,product_rank_in_aisle,aisle_candidate_rank
1,12916,Ginger Ale,1,2,1
1,10957,Fridge Pack Cola,1,3,2
1,47141,Cola,1,4,3
1,16696,Coke Classic,1,5,4
1,40910,Root Beer,1,6,5
1,4138,Arancita Rossa,1,7,6
1,24759,Club Soda,1,8,7
1,44375,Canned Aranciata Orange,1,9,8
1,16290,Sparkling Clementine Juice,1,10,9
1,3599,Baked Aged White Cheddar Rice and Corn Puffs,2,1,10


### 10.2 — Rank Co-Purchase New Candidates

We rank the co-purchase candidates for each customer.

Candidates receive higher priority when:

- More previously purchased products support the recommendation
- The strongest association has higher confidence
- The relationship has higher lift
- The relationship is supported by more historical co-purchase baskets

This ranking allows us to keep only the strongest personalized co-purchase candidates later.

In [0]:
copurchase_candidate_rank_window = (
    Window
    .partitionBy("user_id")
    .orderBy(
        F.desc("supporting_source_products"),
        F.desc("max_confidence"),
        F.desc("max_lift"),
        F.desc("max_copurchase_orders"),
        F.asc("product_id")
    )
)

ranked_copurchase_candidates_df = (
    new_copurchase_candidates_df
    .withColumn(
        "copurchase_candidate_rank",
        F.row_number().over(copurchase_candidate_rank_window)
    )
)

display(
    ranked_copurchase_candidates_df
    .select(
        "user_id",
        "product_id",
        "product_name",
        "supporting_source_products",
        F.round("max_confidence", 4).alias("max_confidence"),
        F.round("max_lift", 2).alias("max_lift"),
        "copurchase_candidate_rank"
    )
    .orderBy(
        "user_id",
        "copurchase_candidate_rank"
    )
    .limit(30)
)

user_id,product_id,product_name,supporting_source_products,max_confidence,max_lift,copurchase_candidate_rank
1,6184,Clementines,9,0.2185,22.77,1
1,24852,Banana,6,0.3891,2.65,2
1,37710,Trail Mix,6,0.185,51.24,3
1,21137,Organic Strawberries,6,0.174,2.12,4
1,47209,Organic Hass Avocado,5,0.1653,2.49,5
1,21903,Organic Baby Spinach,4,0.1883,2.51,6
1,16797,Strawberries,2,0.1663,3.77,7
1,12341,Hass Avocados,2,0.1391,8.91,8
1,43352,Raspberries,2,0.1246,7.02,9
1,47766,Organic Avocado,2,0.1241,2.25,10


### 10.3 — Test a More Compact Candidate Set

Keeping every generated candidate nearly doubles the size of the original candidate set.

We therefore test a more selective strategy:

- Top 20 aisle-based new candidates per customer
- Top 10 co-purchase new candidates per customer

We then measure whether this smaller candidate set preserves most of the coverage improvement.

In [0]:
# Keep Top 20 aisle-based candidates
top20_aisle_candidates_df = (
    ranked_aisle_candidates_df
    .filter(F.col("aisle_candidate_rank") <= 20)
    .select(
        "user_id",
        "product_id"
    )
)

# Keep Top 10 co-purchase candidates
top10_copurchase_candidates_df = (
    ranked_copurchase_candidates_df
    .filter(F.col("copurchase_candidate_rank") <= 10)
    .select(
        "user_id",
        "product_id"
    )
)

# Combine both new-product strategies
pruned_new_candidates_df = (
    top20_aisle_candidates_df
    .unionByName(top10_copurchase_candidates_df)
    .distinct()
)

# Add target order ID
pruned_new_candidates_with_target_df = (
    pruned_new_candidates_df
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id"
        ),
        on="user_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
)

# Combine with original reorder candidates
pruned_final_candidates_df = (
    baseline_candidates_df
    .unionByName(pruned_new_candidates_with_target_df)
    .distinct()
)

# Candidate set size
pruned_candidate_count = pruned_final_candidates_df.count()

# Target products recovered
pruned_recovered_count = (
    target_products_df
    .join(
        pruned_final_candidates_df,
        on=[
            "user_id",
            "target_order_id",
            "product_id"
        ],
        how="inner"
    )
    .count()
)

pruned_coverage = (
    pruned_recovered_count
    / total_target_products
    * 100
)

avg_candidates = (
    pruned_candidate_count
    / 131209
)

print("Candidate pairs:", pruned_candidate_count)
print(
    "Average candidates per customer:",
    round(avg_candidates, 2)
)
print(
    "Target-basket coverage:",
    round(pruned_coverage, 2),
    "%"
)
print(
    "Improvement over baseline:",
    round(pruned_coverage - baseline_coverage, 2),
    "percentage points"
)

Candidate pairs: 11913155
Average candidates per customer: 90.8
Target-basket coverage: 62.52 %
Improvement over baseline: 2.66 percentage points


### 10.4 — Test a Moderate Candidate Budget

The first pruning strategy reduced computational cost but also removed some useful target products.

We therefore test a moderately larger candidate budget:

- Top 30 aisle-based candidates per customer
- Top 15 co-purchase candidates per customer

This helps us evaluate the trade-off between candidate-set size and target-basket coverage.

In [0]:
# Keep Top 30 aisle candidates
top30_aisle_candidates_df = (
    ranked_aisle_candidates_df
    .filter(F.col("aisle_candidate_rank") <= 30)
    .select(
        "user_id",
        "product_id"
    )
)

# Keep Top 15 co-purchase candidates
top15_copurchase_candidates_df = (
    ranked_copurchase_candidates_df
    .filter(F.col("copurchase_candidate_rank") <= 15)
    .select(
        "user_id",
        "product_id"
    )
)

# Combine advanced candidates
moderate_new_candidates_df = (
    top30_aisle_candidates_df
    .unionByName(top15_copurchase_candidates_df)
    .distinct()
)

# Add target order
moderate_new_candidates_with_target_df = (
    moderate_new_candidates_df
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id"
        ),
        on="user_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
)

# Combine with baseline candidates
moderate_final_candidates_df = (
    baseline_candidates_df
    .unionByName(moderate_new_candidates_with_target_df)
    .distinct()
)

# Candidate count
moderate_candidate_count = (
    moderate_final_candidates_df.count()
)

# Target products recovered
moderate_recovered_count = (
    target_products_df
    .join(
        moderate_final_candidates_df,
        on=[
            "user_id",
            "target_order_id",
            "product_id"
        ],
        how="inner"
    )
    .count()
)

moderate_coverage = (
    moderate_recovered_count
    / total_target_products
    * 100
)

moderate_avg_candidates = (
    moderate_candidate_count / 131209
)

print("Candidate pairs:", moderate_candidate_count)

print(
    "Average candidates per customer:",
    round(moderate_avg_candidates, 2)
)

print(
    "Target-basket coverage:",
    round(moderate_coverage, 2),
    "%"
)

print(
    "Improvement over baseline:",
    round(moderate_coverage - baseline_coverage, 2),
    "percentage points"
)

Candidate pairs: 13519765
Average candidates per customer: 103.04
Target-basket coverage: 63.12 %
Improvement over baseline: 3.26 percentage points


### 10.5 — Select the Final Candidate Budget

We compare the candidate-generation configurations to balance target-basket coverage with computational cost.

The `30 aisle + 15 co-purchase` strategy is selected as the final candidate budget.

It increases target-basket coverage from `59.86%` to `63.12%` while avoiding the additional candidate volume required by the fully unpruned strategy.

This provides a practical compromise between recommendation coverage and dataset size.

In [0]:
candidate_budget_comparison_df = spark.createDataFrame(
    [
        (
            "Baseline",
            8474661,
            64.59,
            59.86
        ),
        (
            "Top 20 Aisle + Top 10 Co-purchase",
            11913155,
            90.80,
            62.52
        ),
        (
            "Top 30 Aisle + Top 15 Co-purchase",
            13519765,
            103.04,
            63.12
        ),
        (
            "All Advanced Candidates",
            15665871,
            119.40,
            63.69
        )
    ],
    [
        "strategy",
        "candidate_pairs",
        "avg_candidates_per_customer",
        "target_basket_coverage"
    ]
)

display(candidate_budget_comparison_df)

strategy,candidate_pairs,avg_candidates_per_customer,target_basket_coverage
Baseline,8474661,64.59,59.86
Top 20 Aisle + Top 10 Co-purchase,11913155,90.8,62.52
Top 30 Aisle + Top 15 Co-purchase,13519765,103.04,63.12
All Advanced Candidates,15665871,119.4,63.69


### 10.6 — Add Candidate Generation Sources

Each candidate can originate from a different recommendation strategy.

We create indicators showing whether a product was generated as:

- A historical reorder candidate
- A favorite-aisle candidate
- A co-purchase candidate

For new products, a candidate may be supported by both the aisle and co-purchase strategies.

These source indicators can later be used as additional machine learning features.

In [0]:
# Baseline reorder candidates
baseline_source_df = (
    baseline_candidates_df
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
    .withColumn("is_reorder_candidate", F.lit(1))
    .withColumn("is_aisle_candidate", F.lit(0))
    .withColumn("is_copurchase_candidate", F.lit(0))
    .withColumn("aisle_candidate_rank", F.lit(None).cast("int"))
    .withColumn("copurchase_candidate_rank", F.lit(None).cast("int"))
)

# Top 30 aisle candidates
aisle_source_df = (
    ranked_aisle_candidates_df
    .filter(F.col("aisle_candidate_rank") <= 30)
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id"
        ),
        on="user_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id",
        "aisle_candidate_rank"
    )
    .withColumn("is_reorder_candidate", F.lit(0))
    .withColumn("is_aisle_candidate", F.lit(1))
    .withColumn("is_copurchase_candidate", F.lit(0))
    .withColumn("copurchase_candidate_rank", F.lit(None).cast("int"))
)

# Top 15 co-purchase candidates
copurchase_source_df = (
    ranked_copurchase_candidates_df
    .filter(F.col("copurchase_candidate_rank") <= 15)
    .join(
        target_orders_df.select(
            "user_id",
            "target_order_id"
        ),
        on="user_id",
        how="inner"
    )
    .select(
        "user_id",
        "target_order_id",
        "product_id",
        "copurchase_candidate_rank"
    )
    .withColumn("is_reorder_candidate", F.lit(0))
    .withColumn("is_aisle_candidate", F.lit(0))
    .withColumn("is_copurchase_candidate", F.lit(1))
    .withColumn("aisle_candidate_rank", F.lit(None).cast("int"))
)

# Combine all candidate sources
candidate_sources_df = (
    baseline_source_df
    .unionByName(aisle_source_df)
    .unionByName(copurchase_source_df)
    .groupBy(
        "user_id",
        "target_order_id",
        "product_id"
    )
    .agg(
        F.max("is_reorder_candidate").alias("is_reorder_candidate"),
        F.max("is_aisle_candidate").alias("is_aisle_candidate"),
        F.max("is_copurchase_candidate").alias("is_copurchase_candidate"),
        F.min("aisle_candidate_rank").alias("aisle_candidate_rank"),
        F.min("copurchase_candidate_rank").alias("copurchase_candidate_rank")
    )
    .withColumn(
        "candidate_source_count",
        F.col("is_reorder_candidate")
        + F.col("is_aisle_candidate")
        + F.col("is_copurchase_candidate")
    )
    .withColumn(
        "candidate_source",
        F.when(
            F.col("is_reorder_candidate") == 1,
            "reorder"
        )
        .when(
            (F.col("is_aisle_candidate") == 1) &
            (F.col("is_copurchase_candidate") == 1),
            "aisle+copurchase"
        )
        .when(
            F.col("is_aisle_candidate") == 1,
            "aisle"
        )
        .when(
            F.col("is_copurchase_candidate") == 1,
            "copurchase"
        )
    )
)

display(
    candidate_sources_df
    .orderBy(
        "user_id",
        "candidate_source",
        "product_id"
    )
    .limit(30)
)

user_id,target_order_id,product_id,is_reorder_candidate,is_aisle_candidate,is_copurchase_candidate,aisle_candidate_rank,copurchase_candidate_rank,candidate_source_count,candidate_source
1,1187899,248,0,1,0,25,null,1,aisle
1,1187899,1700,0,1,0,14,null,1,aisle
1,1187899,3599,0,1,0,10,null,1,aisle
1,1187899,4138,0,1,0,6,null,1,aisle
1,1187899,5134,0,1,0,23,null,1,aisle
1,1187899,6448,0,1,0,17,null,1,aisle
1,1187899,7179,0,1,0,28,null,1,aisle
1,1187899,7503,0,1,0,20,null,1,aisle
1,1187899,9755,0,1,0,12,null,1,aisle
1,1187899,10957,0,1,0,2,null,1,aisle


### 10.7 — Analyze Candidate Sources

We summarize how the final candidate set was generated.

This shows how many candidates come from:

- Historical reorders
- Favorite aisles
- Co-purchase relationships
- Both aisle and co-purchase strategies

Understanding this distribution helps evaluate how much each candidate-generation strategy contributes to the final system.

In [0]:
candidate_source_summary_df = (
    candidate_sources_df
    .groupBy("candidate_source")
    .agg(
        F.count("*").alias("candidate_pairs"),
        F.countDistinct("user_id").alias("customers")
    )
    .withColumn(
        "percentage",
        F.round(
            F.col("candidate_pairs")
            / F.sum("candidate_pairs").over(Window.partitionBy())
            * 100,
            2
        )
    )
    .orderBy(F.desc("candidate_pairs"))
)

display(candidate_source_summary_df)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


candidate_source,candidate_pairs,customers,percentage
reorder,8474661,131209,62.68
aisle,3256174,131209,24.08
copurchase,1144186,130517,8.46
aisle+copurchase,644744,115122,4.77


### 10.8 — Validate the Final Candidate Dataset

Before saving the candidate dataset, we verify that each customer-product-target order combination appears only once.

A duplicate count of `0` confirms that the final candidate table is uniquely defined.

In [0]:
final_candidate_validation_df = (
    candidate_sources_df
    .agg(
        F.count("*").alias("total_rows"),
        F.countDistinct(
            "user_id",
            "target_order_id",
            "product_id"
        ).alias("unique_candidate_rows")
    )
    .withColumn(
        "duplicate_rows",
        F.col("total_rows") - F.col("unique_candidate_rows")
    )
)

display(final_candidate_validation_df)

total_rows,unique_candidate_rows,duplicate_rows
13519765,13519765,0


### 10.9 — Add the Next-Basket Target Label

The expanded candidate set contains both previously purchased products and new-to-customer products.

Therefore, the prediction target is now defined as:

- `1` — the candidate product appears in the customer's target basket
- `0` — the candidate product does not appear in the target basket

We use the name `target_purchased` because the task now goes beyond reorder prediction toward full next-basket prediction.

In [0]:
# Create positive labels from actual target-basket products
target_labels_df = (
    target_products_df
    .select(
        "user_id",
        "target_order_id",
        "product_id"
    )
    .withColumn(
        "target_purchased",
        F.lit(1)
    )
)

# Attach labels to the final candidate set
final_labeled_candidates_df = (
    candidate_sources_df
    .join(
        target_labels_df,
        on=[
            "user_id",
            "target_order_id",
            "product_id"
        ],
        how="left"
    )
    .fillna({
        "target_purchased": 0
    })
)

# Check target distribution
display(
    final_labeled_candidates_df
    .groupBy("target_purchased")
    .count()
    .withColumn(
        "percentage",
        F.round(
            F.col("count")
            / F.sum("count").over(Window.partitionBy())
            * 100,
            2
        )
    )
    .orderBy("target_purchased")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


target_purchased,count,percentage
0,12645845,93.54
1,873920,6.46


### 10.10 — Save the Final Advanced Candidate Dataset

We save the final labeled candidate set as a Delta table.

The dataset contains:

- Historical reorder candidates
- Favorite-aisle new-product candidates
- Co-purchase new-product candidates
- Candidate-source indicators and ranks
- The `target_purchased` label

This table will serve as the foundation for the next-basket prediction model.

In [0]:
(
    final_labeled_candidates_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.ml_data.next_basket_candidates"
    )
)

print("Advanced candidate dataset saved successfully:")
print("workspace.ml_data.next_basket_candidates")

Advanced candidate dataset saved successfully:
workspace.ml_data.next_basket_candidates


### 10.11 — Verify the Saved Candidate Table

We reload the saved Delta table and verify its size and target distribution.

This confirms that the final advanced candidate dataset was persisted correctly and is ready for the next modeling notebook.

In [0]:
saved_candidates_df = spark.table(
    "workspace.ml_data.next_basket_candidates"
)

saved_summary_df = (
    saved_candidates_df
    .agg(
        F.count("*").alias("saved_rows"),
        F.countDistinct("user_id").alias("saved_customers"),
        F.sum("target_purchased").alias("positive_targets")
    )
)

display(saved_summary_df)

saved_rows,saved_customers,positive_targets
13519765,131209,873920


## 11. Conclusion

In this notebook, we expanded the recommendation system beyond previously purchased products by developing an advanced candidate-generation framework for next-basket prediction.

Three candidate sources were combined:

- Historical reorder candidates
- Popular products from customers' preferred aisles
- Products discovered through historical co-purchase relationships

The original reorder-only candidate strategy covered:

- **59.86%** of products appearing in customers' target baskets.

The advanced candidate-generation strategies improved coverage to:

- **63.12%** using the selected Top 30 aisle + Top 15 co-purchase configuration.

The selected configuration contains:

- **13,519,765 candidate customer-product pairs**
- **131,209 customers**
- Approximately **103 candidates per customer**
- **873,920 positive target products**

The candidate set contains:

- **62.68%** historical reorder candidates
- **24.08%** aisle-based candidates
- **8.46%** co-purchase candidates
- **4.77%** candidates supported by both aisle and co-purchase strategies

Aisle-based generation and co-purchase generation were complementary: combining both recovered more new-to-customer products than either method alone.

The final dataset was labeled using `target_purchased`, where:

- `1` indicates that the candidate appeared in the customer's next basket.
- `0` indicates that it did not.

The final candidate dataset was saved as:

`workspace.ml_data.next_basket_candidates`

This dataset provides the foundation for the next stage of the project: engineering features for the expanded candidate set and training a true next-basket prediction model capable of scoring both reorder and new-to-customer products.